# Encoder from Scratch

### Check version

In [18]:
from importlib.metadata import version
print("torch version: ", version("torch"))

torch version:  2.7.0+cu128


### Previous implementations

Multi-Head Attention, Feed Forward Network, LayerNorm

In [19]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

from dataclasses import dataclass

import math

In [20]:
# form a dataclass ModelArgs
@dataclass
class ModelArgs:
    max_T: int
    D: int
    H: int 
    hidden_D: int
    dropout: float
    n_layers: int

In [21]:
# MultiHeadAttention
class MultiHeadAttention(nn.Module):
    
    def __init__(self, args: ModelArgs, is_causal = False):
        # inherit from nn.Module
        super().__init__()

        # set parameters
        self.H = args.H
        self.D = args.D
        assert self.D % self.H == 0
        self.D_h = self.D // self.H # D_h is each head embedding dimension
        self.is_causal = is_causal
        
        # linear transformtion layers
        # shape for Wq, Wk, Wv, Wo = [D, D]
        self.Wq = nn.Linear(self.D, self.D, bias = False)
        self.Wk = nn.Linear(self.D, self.D, bias = False)
        self.Wv = nn.Linear(self.D, self.D, bias = False)
        self.Wo = nn.Linear(self.D, self.D, bias = False)

        # dropout layer
        self.res_dropout = nn.Dropout(args.dropout)

        # mask
        if is_causal:
            mask = torch.full(
                (1, 1, args.max_T, args.max_T), 
                float("-inf"))
            mask = torch.triu(mask, diagonal=1)
            self.register_buffer("mask", mask) # register_buffer(name, tensor) tells the program this is not a model parameter
    
    def forward(self, q, k, v):
        # q, k, v shape = [B, T, D]
        B, T = q.shape[0], q.shape[1] 
        D = self.D
        H, D_h = self.H, self.D_h

        # step 1: linear transformtion with W
        # [B, T, D] * [D, D] -> [B, T, D]
        # shape for Q, K, V = [B, T, D]
        Q = self.Wq(q)
        K = self.Wk(k)
        V = self.Wv(v)

        # step 2: split into multiple heads
        # [B, T, D] = [B, T, H, D_h] where D = H * D_h
        # shape for Q, K, V = [B, T, H, D_h]
        Q = Q.view(B, T, H, D_h)
        K = K.view(B, T, H, D_h)
        V = V.view(B, T, H, D_h)

        # step 3: switch positions for T and H
        # [B, T, H, D_h] -> [B, H, T, D_h]
        # shape for Q, K, V = [B, H, T, D_h]
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # step 4: compute attention score S
        # S = (Q K^T) / sqrt(D_h)
        # shape for S = [B, H, T, D_h] * [B, H, D_h, T] -> [B, H, T, T]
        S = torch.matmul(Q, K.transpose(2, 3)) / math.sqrt(D_h)

        # step 5: mask
        # shape for S = [B, H, T, T]
        if self.is_causal:
            S = S + self.mask[:, :, :T, :T]

        # step 6: compute attention probabiloity S_prob by softmax
        # normalize the last dimension T
        # shape for S_prob = [B, H, T, T]
        S_prob = F.softmax(S, dim=-1).type_as(Q)

        # step 7: compute weighted V
        # S_prob * V -> [B, H, T, T] * [B, H, T, D_h] -> [B, H, T, D_h]
        # shape for output: [B, H, T, D_h]
        output = torch.matmul(S_prob, V)

        # step 8: concatenate multiple heads
        # [B, H, T, D_h] -> [B, T, H, D_h] -> [B, T, D]
        # shape for output: [B, T, D]
        output = output.transpose(1, 2).contiguous().view(B, T, D)

        # step 9: final linear transformation layer + residual dropout
        # [B, T, D] * [D, D] -> [B, T, D]
        # shape for output: [B, T, D]
        output = self.Wo(output)
        output = self.res_dropout(output)

        return output

In [22]:
# FFN
class FFN(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.w1 = nn.Linear(args.D, args.hidden_D, bias=False)
        self.w2 = nn.Linear(args.hidden_D, args.D, bias=False)
        self.dropout = nn.Dropout(args.dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.relu(self.w1(x))))

In [23]:
# LayerNorm
class LayerNorm(nn.Module):
    """
    formula:
    output = ((x - mean(x)) / (std(x) + eps)) * gamma + beta
    gamma and beta are learned parameters
    """
    def __init__(self, args: ModelArgs, eps=1e-6):
        super().__init__()
        
        # define gamma and beta
        self.gamma = nn.Parameter(torch.ones(args.D))
        self.beta = nn.Parameter(torch.zeros(args.D))

        self.eps = eps

    def forward(self, x):
        # x shape = [B, T, D], normalize within the last dimension D
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)

        return (x - mean) / (std + self.eps) * self.gamma + self.beta

### Implementation for Encoder

In [24]:
# encoder layer
class EncoderLayer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()

        self.norm1 = LayerNorm(args)
        self.attention = MultiHeadAttention(args, is_causal=False)

        self.norm2 = LayerNorm(args)
        self.ffn = FFN(args)

    def forward(self, x):
        x = self.norm1(x)
        x = x + self.attention(x, x, x)
        x = self.norm2(x)
        output = x + self.ffn(x)
        return output

In [25]:
# encoder
class Encoder(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()

        # multiple layers
        self.layers = nn.ModuleList([EncoderLayer(args) for _ in range(args.n_layers)])

        self.norm = LayerNorm(args)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        
        output = self.norm(x)
        return output

### Test

In [26]:
# hyperparameters
B = 10
H = 8
D = 768
hidden_D = D * 4
dropout = 0.1
max_T = 512
n_layers = 6

# instantiate args
args = ModelArgs(max_T = max_T, D = D, H = H, hidden_D = hidden_D, dropout = dropout, n_layers = n_layers)

# input
x = torch.randn(B, max_T, D)

# test encoder
encoder = Encoder(args)
encoder_output = encoder(x)

print("encoder output shape: ", encoder_output.shape)
print("encoder output: ", encoder_output)


encoder output shape:  torch.Size([10, 512, 768])
encoder output:  tensor([[[ 3.6514e-01, -1.0852e+00,  4.5850e-02,  ...,  2.7647e+00,
           1.2695e+00, -4.4521e-01],
         [ 5.1574e-01,  3.3490e-01, -2.2885e-01,  ...,  1.3247e+00,
          -4.7731e-01, -1.1060e+00],
         [ 3.0307e-02,  5.9327e-03, -3.2726e-01,  ..., -4.5581e-02,
          -1.9714e-01,  1.3157e-01],
         ...,
         [-1.5089e-01,  1.7272e+00,  1.8199e+00,  ..., -3.5047e-01,
           9.0779e-01, -1.0381e+00],
         [-1.2600e+00, -1.0200e+00, -8.0495e-03,  ...,  5.1183e-01,
           6.1650e-01,  6.5956e-01],
         [ 1.1165e+00, -4.8715e-01, -2.6887e-01,  ...,  8.6567e-01,
           2.4853e-01,  2.6919e+00]],

        [[ 3.4319e-01, -3.8508e-01,  1.8071e-03,  ...,  1.1874e+00,
           1.1598e+00, -5.9661e-02],
         [ 1.4466e+00, -1.6232e+00,  6.2276e-03,  ...,  1.1157e+00,
          -1.8776e-04,  1.0937e+00],
         [ 9.1994e-01, -2.2325e+00, -2.6752e-01,  ...,  1.0836e+00,
         